# Fruit Quality Grading
## Step 2: CNN Training with EfficientNetB0 (v4 - Underfitting Fix)

### Why v4
The v3 run underfit badly: accuracy stayed close to random guessing for a 9-class problem. The likely causes were learning rates that were too small, dropout that was too high, overly aggressive augmentation/noise, and image scaling that may not match EfficientNetB0's expected input range.

### Changes from v3
| Issue | v4 Fix Applied |
|---|---|
| Phase 2/3 learning too slowly | Phase 2 LR restored to **1e-5**, Phase 3 LR set to **1e-6** |
| Too much regularization | Dropout reduced to **0.4 / 0.3** |
| Augmentation too destructive | Removed Gaussian noise, removed vertical flip, reduced jitter/rotation/zoom strength |
| EfficientNet input scale risk | Feed EfficientNet native **0..255** float images instead of dividing by 255 |
| Need to preserve previous artifacts | Saves checkpoints, plots, reports, Grad-CAM, and final model with **v4** filenames |

### Pipeline
```
output/labeled_dataset.csv  (from Step 1)
           ->
  Load & verify labels
           ->
  Train / Validation / Test split (stratified, 70/15/15)
           ->
  Moderate data augmentation (training only)
           ->
  Phase 1 : Train head only        (frozen base,     LR=1e-3, 20 epochs)
           ->
  Phase 2 : Unfreeze top 20 layers (partial unfreeze, LR=1e-5, 25 epochs)
           ->
  Phase 3 : Full fine-tune         (all layers,       LR=1e-6, 35 epochs)
           ->
  Evaluate - Accuracy, F1, Confusion Matrix
           ->
  Grad-CAM visualizations
           ->
  Save model -> output/models/fruit_grader_v4.keras
```


## 📦 1. Install Dependencies

In [ ]:
!pip install tensorflow scikit-learn pandas matplotlib seaborn opencv-python tqdm -q
print("All dependencies installed")

## 📚 2. Imports

In [ ]:
import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {'Yes — ' + gpus[0].name if gpus else 'No (CPU)'}")

## ⚙️ 3. Configuration

Key changes from v1: lower learning rates, 3-phase training, stronger regularization.

In [ ]:
# ---------------------------------------------
#  CONFIGURATION
# ---------------------------------------------

LABELED_CSV  = "output/labeled_dataset.csv"
OUTPUT_DIR   = "output"
MODEL_DIR    = "output/models"
RUN_VERSION  = "v4"

IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 32
RANDOM_STATE = 42

# EfficientNetB0 in tf.keras includes its own preprocessing/rescaling.
# Keep images in the native 0..255 float range for the pretrained backbone.
USE_EFFICIENTNET_NATIVE_SCALE = True

# Three-phase training (v4: recover learning capacity after v3 underfit)
# Phase 1: head only, base fully frozen
PHASE1_EPOCHS = 20
PHASE1_LR     = 1e-3

# Phase 2: unfreeze only top 20 layers of base
PHASE2_EPOCHS = 25
PHASE2_LR     = 1e-5

# Phase 3: unfreeze full network with a smaller-but-still-learning LR
PHASE3_EPOCHS = 35
PHASE3_LR     = 1e-6

# Moderate regularization
DROPOUT_1    = 0.4
DROPOUT_2    = 0.3
DENSE_L2     = 1e-4
WEIGHT_DECAY = 0.0

# Split
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15

# Classes
FRUITS      = ["apple", "orange", "banana"]
GRADES      = ["A", "B", "C"]
CLASS_NAMES = [f"{f}_{g}" for f in FRUITS for g in GRADES]

for d in [MODEL_DIR, f"{OUTPUT_DIR}/plots", f"{OUTPUT_DIR}/gradcam"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")
print(f"Run version: {RUN_VERSION}")
print(f"EfficientNet native input scale: {USE_EFFICIENTNET_NATIVE_SCALE}")
print(f"Training phases: 3  |  Total max epochs: {PHASE1_EPOCHS+PHASE2_EPOCHS+PHASE3_EPOCHS}")

## 📂 4. Load & Verify Labeled Dataset

In [ ]:
df = pd.read_csv(LABELED_CSV)
df["exists"] = df["image_path"].apply(lambda p: Path(p).exists())
missing = df[~df["exists"]]
if len(missing) > 0:
    print(f"WARNING: {len(missing)} missing paths dropped.")
    df = df[df["exists"]].copy()
df = df.reset_index(drop=True)

print(f"Total images : {len(df):,}")
print(f"\nLabel distribution:")
for label, count in df["label"].value_counts().sort_index().items():
    bar = chr(9608) * (count // 30)
    print(f"  {label:12s}  {count:>5,}  {bar}")

## ✂️ 5. Stratified Train / Val / Test Split

In [ ]:
df_train, df_temp = train_test_split(
    df, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df["label"], random_state=RANDOM_STATE
)
val_frac = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
df_val, df_test = train_test_split(
    df_temp, test_size=(1 - val_frac),
    stratify=df_temp["label"], random_state=RANDOM_STATE
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Train : {len(df_train):,}  ({len(df_train)/len(df)*100:.0f}%)")
print(f"Val   : {len(df_val):,}  ({len(df_val)/len(df)*100:.0f}%)")
print(f"Test  : {len(df_test):,}  ({len(df_test)/len(df)*100:.0f}%)")

## ⚖️ 6. Class Weights

Since K-Means may produce unequal cluster sizes, we compute class weights so the model doesn't bias toward majority grades.

In [ ]:
le = LabelEncoder()
le.fit(CLASS_NAMES)
NUM_CLASSES = len(le.classes_)

y_train = le.transform(df_train["label"].values)
y_val   = le.transform(df_val["label"].values)
y_test  = le.transform(df_test["label"].values)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights_array))

print("Class weights (higher = minority class gets more attention):")
for idx, weight in class_weight_dict.items():
    label = le.inverse_transform([idx])[0]
    bar   = chr(9608) * int(weight * 10)
    print(f"  {label:12s}  {weight:.3f}  {bar}")

## 🔄 7. Data Pipeline & Augmentation

**Stronger augmentation than v1** — essential for a small dataset (~500 images/class).

In [ ]:
def load_and_preprocess(path, augment=False):
    """Load, resize, and optionally augment an image."""
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMAGE_SIZE)
    img = img.astype(np.float32)

    if not USE_EFFICIENTNET_NATIVE_SCALE:
        img = img / 255.0

    max_value = 255.0 if USE_EFFICIENTNET_NATIVE_SCALE else 1.0

    if augment:
        # Horizontal flip
        if np.random.rand() > 0.5:
            img = np.fliplr(img)
        # Moderate brightness jitter
        img = np.clip(img * np.random.uniform(0.8, 1.2), 0, max_value)
        # Moderate contrast jitter
        mean = img.mean()
        img = np.clip((img - mean) * np.random.uniform(0.85, 1.15) + mean, 0, max_value)
        # Moderate rotation
        angle = np.random.uniform(-15, 15)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
        # Occasional random zoom: crop and resize back
        if np.random.rand() > 0.5:
            zoom = np.random.uniform(0.9, 1.0)
            ch, cw = int(h * zoom), int(w * zoom)
            y1 = np.random.randint(0, h - ch + 1)
            x1 = np.random.randint(0, w - cw + 1)
            img = img[y1:y1 + ch, x1:x1 + cw]
            img = cv2.resize(img, IMAGE_SIZE)

    return img.astype(np.float32)


def make_dataset(df_split, labels, augment=False, shuffle=False):
    """Build a tf.data.Dataset from a DataFrame."""
    paths = df_split["image_path"].values
    y_onehot = tf.keras.utils.to_categorical(labels, NUM_CLASSES)

    def generator():
        indices = np.arange(len(paths))
        if shuffle:
            np.random.shuffle(indices)
        for i in indices:
            img = load_and_preprocess(paths[i], augment=augment)
            yield img, y_onehot[i]

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(*IMAGE_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32),
        )
    )
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(df_train, y_train, augment=True, shuffle=True)
val_ds = make_dataset(df_val, y_val, augment=False, shuffle=False)
test_ds = make_dataset(df_test, y_test, augment=False, shuffle=False)

print("Datasets ready:")
print(f"  train_ds : {len(df_train):,} images | moderate augmentation ON")
print(f"  val_ds   : {len(df_val):,} images  | augmentation OFF")
print(f"  test_ds  : {len(df_test):,} images  | augmentation OFF")

## 🖼️ 8. Augmentation Preview

In [ ]:
# Show one image with 8 augmented versions
sample_path = df_train.sample(1, random_state=7)["image_path"].values[0]
original    = load_and_preprocess(sample_path, augment=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0, 0].imshow(original)
axes[0, 0].set_title("Original", fontsize=10, fontweight="bold")
axes[0, 0].axis("off")

for i, ax in enumerate(axes.flat[1:]):
    aug = load_and_preprocess(sample_path, augment=True)
    ax.imshow(aug)
    ax.set_title(f"Augmented {i+1}", fontsize=9)
    ax.axis("off")

plt.suptitle("Augmentation Examples", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/augmentation_preview.png", dpi=120, bbox_inches="tight")
plt.show()

## 🧠 9. Build Model — EfficientNetB0

**Key fix from v1**: Higher dropout (0.5) and L2 regularization added to the dense layer.

In [ ]:
def make_optimizer(learning_rate, weight_decay):
    """Create AdamW only when weight decay is enabled; otherwise use Adam."""
    if weight_decay and weight_decay > 0:
        if hasattr(keras.optimizers, "AdamW"):
            return keras.optimizers.AdamW(
                learning_rate=learning_rate,
                weight_decay=weight_decay,
            )
        if hasattr(keras.optimizers, "experimental") and hasattr(keras.optimizers.experimental, "AdamW"):
            return keras.optimizers.experimental.AdamW(
                learning_rate=learning_rate,
                weight_decay=weight_decay,
            )
        print("WARNING: AdamW not available; falling back to Adam without optimizer weight decay.")
    return keras.optimizers.Adam(learning_rate=learning_rate)


def build_model(
    num_classes,
    learning_rate,
    freeze_base=True,
    unfreeze_top_n=0,
    dropout_1=DROPOUT_1,
    dropout_2=DROPOUT_2,
    weight_decay=WEIGHT_DECAY,
):
    """
    Build EfficientNetB0 model.
    freeze_base=True  : freeze entire base
    unfreeze_top_n>0  : unfreeze last N layers of base
    freeze_base=False : unfreeze full base
    """
    base = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(*IMAGE_SIZE, 3),
    )

    # Freeze strategy
    if freeze_base:
        base.trainable = False
    elif unfreeze_top_n > 0:
        for layer in base.layers[:-unfreeze_top_n]:
            layer.trainable = False
        for layer in base.layers[-unfreeze_top_n:]:
            layer.trainable = True
    else:
        base.trainable = True

    # Classification head (v4: moderate dropout + L2)
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_1)(x)
    x = layers.Dense(
        256, activation="relu",
        kernel_regularizer=keras.regularizers.l2(DENSE_L2)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=make_optimizer(learning_rate, weight_decay),
        loss="categorical_crossentropy",
        metrics=["accuracy",
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall")],
    )
    return model, base


model, base_model = build_model(NUM_CLASSES, PHASE1_LR, freeze_base=True)

trainable = sum([tf.size(v).numpy() for v in model.trainable_variables])
total = model.count_params()
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  (head only, base frozen)")
print(f"Dropout          : {DROPOUT_1:.2f} / {DROPOUT_2:.2f}")
print(f"Weight decay     : {WEIGHT_DECAY:g}")
model.summary()

## 🔧 10. Shared Callbacks Helper

In [ ]:
def checkpoint_path(phase: int):
    return f"{MODEL_DIR}/best_phase{phase}_{RUN_VERSION}.keras"


def get_callbacks(phase: int, patience_es=15, patience_lr=6):
    """Return callbacks for a given training phase."""
    return [
        EarlyStopping(
            monitor="val_accuracy", patience=patience_es,
            restore_best_weights=True, verbose=1
        ),
        ModelCheckpoint(
            checkpoint_path(phase),
            monitor="val_accuracy", save_best_only=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss", factor=0.3,
            patience=patience_lr, min_lr=1e-8, verbose=1
        ),
    ]

print("Callbacks ready")

## 🏋️ 11. Phase 1 — Train Head Only (Base Frozen)

The EfficientNetB0 backbone is completely frozen. Only the classification head learns.
This establishes a good starting point before we touch the pretrained weights.

In [ ]:
print("=" * 50)
print("  PHASE 1: Head only | Base frozen")
print(f"  Epochs: {PHASE1_EPOCHS} | LR: {PHASE1_LR}")
print("=" * 50)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=get_callbacks(phase=1),
    class_weight=class_weight_dict,
    verbose=1,
)

best_p1 = max(history1.history["val_accuracy"])
print(f"\nBest val accuracy (Phase 1): {best_p1:.4f}  ({best_p1*100:.2f}%)")

## 🔓 12. Phase 2 — Partial Unfreeze (Top 20 Layers)

**KEY FIX from v1**: Instead of unfreezing the entire network at once, we only unfreeze the last 20 layers of EfficientNetB0. This is much gentler and prevents the wild validation swings seen in v1.

Learning rate is reduced to `1e-5` (was `1e-4` in v1).

In [ ]:
# Rebuild model with top 20 layers unfrozen
model, base_model = build_model(
    NUM_CLASSES, PHASE2_LR,
    freeze_base=False, unfreeze_top_n=20
)

# Load best weights from Phase 1
model.load_weights(checkpoint_path(1))

trainable = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"Trainable params now: {trainable:,}  (head + top 20 base layers)")
print()
print("=" * 50)
print("  PHASE 2: Partial unfreeze (top 20 layers)")
print(f"  Epochs: {PHASE2_EPOCHS} | LR: {PHASE2_LR}")
print("=" * 50)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=get_callbacks(phase=2, patience_es=12, patience_lr=5),
    class_weight=class_weight_dict,
    verbose=1,
)

best_p2 = max(history2.history["val_accuracy"])
print(f"\nBest val accuracy (Phase 2): {best_p2:.4f}  ({best_p2*100:.2f}%)")

## 🔥 13. Phase 3 — Full Fine-Tune (All Layers)

Now we unfreeze the entire network with a very small learning rate (`5e-6`). At this point the model already has good weights from Phase 2, so the full fine-tune only makes small adjustments.

In [ ]:
# Rebuild with full network unfrozen
model, base_model = build_model(
    NUM_CLASSES, PHASE3_LR,
    freeze_base=False, unfreeze_top_n=0
)

# Load best weights from Phase 2
model.load_weights(checkpoint_path(2))

trainable = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"Trainable params now: {trainable:,}  (entire network)")
print()
print("=" * 50)
print("  PHASE 3: Full fine-tune")
print(f"  Epochs: {PHASE3_EPOCHS} | LR: {PHASE3_LR}")
print("=" * 50)

history3 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE3_EPOCHS,
    callbacks=get_callbacks(phase=3, patience_es=15, patience_lr=6),
    class_weight=class_weight_dict,
    verbose=1,
)

best_p3 = max(history3.history["val_accuracy"])
print(f"\nBest val accuracy (Phase 3): {best_p3:.4f}  ({best_p3*100:.2f}%)")

## 📈 14. Training Curves — All 3 Phases

In [ ]:
def merge_histories(*histories):
    merged = {}
    for h in histories:
        for key, vals in h.history.items():
            merged.setdefault(key, []).extend(vals)
    return merged

history = merge_histories(history1, history2, history3)
p1_end  = len(history1.history["accuracy"])
p2_end  = p1_end + len(history2.history["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (train_key, val_key), title in zip(
    axes,
    [("accuracy", "val_accuracy"), ("loss", "val_loss")],
    ["Accuracy", "Loss"]
):
    epochs = range(1, len(history[train_key]) + 1)
    ax.plot(epochs, history[train_key], label=f"Train {title}", linewidth=2, color="#3498db")
    ax.plot(epochs, history[val_key],   label=f"Val {title}",   linewidth=2,
            linestyle="--", color="#e67e22")

    # Phase dividers
    ax.axvline(x=p1_end, color="#95a5a6", linestyle=":", linewidth=1.5)
    ax.axvline(x=p2_end, color="#95a5a6", linestyle=":", linewidth=1.5)

    # Phase labels
    ymin, ymax = ax.get_ylim()
    mid = (ymax + ymin) / 2
    ax.text(p1_end / 2,       ymax * 0.97, "Phase 1", ha="center", fontsize=9, color="#7f8c8d")
    ax.text((p1_end + p2_end) / 2, ymax * 0.97, "Phase 2", ha="center", fontsize=9, color="#7f8c8d")
    ax.text((p2_end + len(epochs)) / 2, ymax * 0.97, "Phase 3", ha="center", fontsize=9, color="#7f8c8d")

    ax.set_title(f"{title} over Epochs (3-Phase Training)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/training_curves_v4.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> output/plots/training_curves_v4.png")

## 🎯 15. Evaluation on Test Set

In [ ]:
phase_val_scores = {
    1: best_p1,
    2: best_p2,
    3: best_p3,
}
best_phase = max(phase_val_scores, key=phase_val_scores.get)
best_val_acc = phase_val_scores[best_phase]

print(f"Best validation checkpoint: Phase {best_phase} ({best_val_acc:.4f}, {best_val_acc*100:.2f}%)")
best_model = keras.models.load_model(checkpoint_path(best_phase))

y_pred_probs = best_model.predict(test_ds, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = y_test

test_loss, test_acc, test_prec, test_rec = best_model.evaluate(test_ds, verbose=0)
f1 = f1_score(y_true, y_pred, average="macro")

print(f"{'='*48}")
print(f"  TEST SET RESULTS (v4 - best overall phase)")
print(f"{'='*48}")
print(f"  Selected phase : {best_phase}")
print(f"  Best val acc   : {best_val_acc:.4f}  ({best_val_acc*100:.2f}%)")
print(f"  Test loss      : {test_loss:.4f}")
print(f"  Accuracy       : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Precision      : {test_prec:.4f}")
print(f"  Recall         : {test_rec:.4f}")
print(f"  F1-Score       : {f1:.4f}  (macro)")
print(f"{'='*48}")


## 📋 16. Per-Class Classification Report

In [ ]:
target_names = le.classes_
report = classification_report(y_true, y_pred, target_names=target_names)
print(report)
with open(f"{OUTPUT_DIR}/classification_report_v4.txt", "w") as f:
    f.write(report)
print("Saved -> output/classification_report_v4.txt")

## 🔢 17. Confusion Matrix

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ["d", ".2f"],
    ["Confusion Matrix (Counts)", "Confusion Matrix (Normalized)"]
):
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=target_names, yticklabels=target_names,
        ax=ax, linewidths=0.5, linecolor="white",
        annot_kws={"size": 9},
    )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True", fontsize=11)
    ax.tick_params(axis="x", rotation=35)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/confusion_matrix_v4.png", dpi=150, bbox_inches="tight")
plt.show()

## 🍎🍊🍌 18. Per-Fruit Per-Grade Accuracy Heatmap

In [ ]:
results = []
for class_name in sorted(CLASS_NAMES):
    class_idx    = le.transform([class_name])[0]
    mask         = (y_true == class_idx)
    if mask.sum() == 0:
        continue
    correct      = (y_pred[mask] == class_idx).sum()
    total        = mask.sum()
    fruit, grade = class_name.split("_")
    results.append({
        "Class":    class_name,
        "Fruit":    fruit.capitalize(),
        "Grade":    grade,
        "Correct":  correct,
        "Total":    total,
        "Accuracy": correct / total,
    })

df_results = pd.DataFrame(results)
print(f"{'Class':<14} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 44)
for _, row in df_results.iterrows():
    bar = chr(9608) * int(row["Accuracy"] * 20)
    print(f"{row['Class']:<14} {int(row['Correct']):>8} {int(row['Total']):>8}  {row['Accuracy']:>8.1%}  {bar}")

pivot = df_results.pivot(index="Fruit", columns="Grade", values="Accuracy")
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt=".1%", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, ax=ax, linewidths=0.5,
            annot_kws={"size": 13, "fontweight": "bold"})
ax.set_title("Per-Fruit Per-Grade Accuracy (v4)", fontsize=13, fontweight="bold")
ax.set_xlabel("Grade")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/per_class_accuracy_v4.png", dpi=150, bbox_inches="tight")
plt.show()

## 🔥 19. Grad-CAM Visualizations

Shows which region of the fruit the model focuses on for grading.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index   = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads       = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap      = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), predictions.numpy()


def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMAGE_SIZE)
    hm  = cv2.applyColorMap(np.uint8(255 * cv2.resize(heatmap, IMAGE_SIZE)), cv2.COLORMAP_JET)
    hm  = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
    return img, (img * (1 - alpha) + hm * alpha).astype(np.uint8)


# Find last conv layer
last_conv = None
for layer in best_model.layers:
    if isinstance(layer, keras.Model):
        for sub in layer.layers:
            if isinstance(sub, layers.Conv2D):
                last_conv = sub.name
if last_conv is None:
    last_conv = "top_conv"
print(f"Using layer: {last_conv}")

# Plot 2 samples per class
N_SHOW = 2
fig, axes = plt.subplots(NUM_CLASSES, N_SHOW * 2, figsize=(4 * N_SHOW * 2, 3.5 * NUM_CLASSES))
grade_colors = {"A": "#2ecc71", "B": "#f39c12", "C": "#e74c3c"}

for row, class_name in enumerate(sorted(CLASS_NAMES)):
    class_idx = le.transform([class_name])[0]
    subset    = df_test[df_test["label"] == class_name]
    samples   = subset.sample(min(N_SHOW, len(subset)), random_state=RANDOM_STATE)
    _, grade  = class_name.split("_")

    for col_pair, (_, sample) in enumerate(samples.iterrows()):
        path = sample["image_path"]
        img_tensor = np.expand_dims(load_and_preprocess(path), axis=0)
        try:
            heatmap, preds = make_gradcam_heatmap(img_tensor, best_model, last_conv)
            pred_label     = le.inverse_transform([np.argmax(preds[0])])[0]
            original, overlay = overlay_gradcam(path, heatmap)

            axes[row, col_pair*2].imshow(original)
            axes[row, col_pair*2].axis("off")
            if col_pair == 0:
                axes[row, col_pair*2].set_ylabel(class_name, fontsize=9, fontweight="bold")

            correct = pred_label == class_name
            axes[row, col_pair*2+1].imshow(overlay)
            axes[row, col_pair*2+1].axis("off")
            axes[row, col_pair*2+1].set_title(
                f"Pred: {pred_label}",
                fontsize=8, fontweight="bold",
                color="#2ecc71" if correct else "#e74c3c"
            )
        except Exception:
            axes[row, col_pair*2].axis("off")
            axes[row, col_pair*2+1].axis("off")

plt.suptitle("Grad-CAM: Original vs Heatmap (green=correct, red=wrong)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/gradcam/gradcam_v4.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved -> output/gradcam/gradcam_v4.png")

## 🔮 20. Predict a Single Image

In [ ]:
def predict_fruit_grade(image_path, model, label_encoder):
    img        = load_and_preprocess(image_path, augment=False)
    preds      = model.predict(np.expand_dims(img, 0), verbose=0)[0]
    pred_idx   = np.argmax(preds)
    pred_label = label_encoder.inverse_transform([pred_idx])[0]
    confidence = preds[pred_idx]
    fruit, grade = pred_label.split("_")

    grade_desc = {"A": "Premium", "B": "Average", "C": "Low"}
    print(f"Fruit      : {fruit.capitalize()}")
    print(f"Grade      : {grade} ({grade_desc[grade]} quality)")
    print(f"Confidence : {confidence:.2%}")
    print("\nTop 3:")
    for idx in np.argsort(preds)[::-1][:3]:
        print(f"  {label_encoder.inverse_transform([idx])[0]:<14} {preds[idx]:.2%}")

    img_disp = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(4, 4))
    plt.imshow(img_disp)
    plt.axis("off")
    plt.title(f"{fruit.capitalize()} — Grade {grade} ({confidence:.1%})",
              fontsize=12, fontweight="bold",
              color={"A": "#2ecc71", "B": "#f39c12", "C": "#e74c3c"}[grade])
    plt.show()

# Try on a random test image
sample_path = df_test.sample(1, random_state=42)["image_path"].values[0]
predict_fruit_grade(sample_path, best_model, le)

## 💾 21. Save Final Model

In [ ]:
best_model.save(f"{MODEL_DIR}/fruit_grader_v4.keras")
np.save(f"{MODEL_DIR}/class_names.npy", le.classes_)

print("=" * 55)
print("  STEP 2 COMPLETE (v4)")
print("=" * 55)
print(f"  Selected Phase: {best_phase}")
print(f"  Best Val Acc  : {best_val_acc:.4f}  ({best_val_acc*100:.2f}%)")
print(f"  Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  F1-Score      : {f1:.4f}  (macro)")
print("")
print("  v4 changes applied:")
print("    EfficientNet native 0..255 image scale")
print("    Phase 2 LR restored to 1e-5")
print("    Phase 3 LR set to 1e-6")
print("    Final evaluation selects the best checkpoint across all phases")
print("    Dropout reduced to 0.4/0.3")
print("    Moderate augmentation without Gaussian noise")
print("    Dense L2 regularization")
print("    Class weights for imbalance")
print("")
print("  Saved:")
print(f"    {MODEL_DIR}/fruit_grader_v4.keras")
print(f"    {MODEL_DIR}/class_names.npy")
print(f"    {OUTPUT_DIR}/plots/training_curves_v4.png")
print(f"    {OUTPUT_DIR}/plots/confusion_matrix_v4.png")
print(f"    {OUTPUT_DIR}/plots/per_class_accuracy_v4.png")
print(f"    {OUTPUT_DIR}/gradcam/gradcam_v4.png")
print("")
print("  Next: Step 3 - Streamlit Demo App")
